In [1]:
#!/usr/bin/env python3
"""
QUANTUM KERNEL TRAINING WITH THERMAL NOISE
Config: 400 Train / 200 Test | Selectable Feature Maps | IBM Eagle Noise Profile
"""

import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix # <--- ADDED confusion_matrix
from sklearn.svm import SVC

# Qiskit Imports
from qiskit import transpile
from qiskit.circuit.library import ZZFeatureMap, ZFeatureMap, PauliFeatureMap
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, thermal_relaxation_error, depolarizing_error, ReadoutError
from qiskit.quantum_info import DensityMatrix, state_fidelity

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
DATA_PATH = 'creditcard.csv'

# --- Feature Selection ---
FEATURE_LIST = ["V14", "V7", "V4", "V19", "V20", "V17"] 

# --- Feature Map Selection ---
# Options: 'Z', 'ZZ', 'Pauli_X_ZZ'
MAP_TYPE = 'ZZ'  
REPS = 1

# --- Dataset Size (Balanced) ---
TRAIN_SIZE = 400
TEST_SIZE = 200

# --- Hardware Noise Profile (IBM Eagle "Lower Quartile") ---
NOISE_CONFIG = {
    'T1': 60.0e3,         # 60 µs
    'T2': 30.0e3,         # 30 µs
    'GATE_TIME_1Q': 60.0, # 60 ns
    'GATE_TIME_2Q': 550.0,# 550 ns
    'P_DEP_1Q': 0.0004,   # 0.04%
    'P_DEP_2Q': 0.02,     # 2.0%
    'P_READOUT': 0.04     # 4.0%
}

print(f"\n{'='*60}")
print(f"QSVM CONFIGURATION")
print(f"{'='*60}")
print(f"Features:    {FEATURE_LIST}")
print(f"Map Type:    {MAP_TYPE}")
print(f"Train Size:  {TRAIN_SIZE}")
print(f"Test Size:   {TEST_SIZE}")
print(f"Simulation:  Density Matrix with Thermal Noise")

# ==============================================================================
# 2. DATA PREPARATION
# ==============================================================================
try:
    df = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    print(f"CRITICAL: {DATA_PATH} not found.")
    exit(1)

X_all = df[FEATURE_LIST].values
y_all = df['Class'].values

# Balance the Dataset (50/50 Fraud/Genuine)
fraud_idx = np.where(y_all == 1)[0]
genuine_idx = np.where(y_all == 0)[0]

# Ensure we have enough data
required_total = TRAIN_SIZE + TEST_SIZE
if len(fraud_idx) < required_total // 2:
    print(f"Warning: Not enough fraud samples for requested size. Using max available.")
    half_size = len(fraud_idx)
else:
    half_size = required_total // 2

indices = np.concatenate([
    np.random.choice(fraud_idx, half_size, replace=False),
    np.random.choice(genuine_idx, half_size, replace=False)
])
np.random.shuffle(indices)

X_bal = X_all[indices]
y_bal = y_all[indices]

# Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, 
    train_size=TRAIN_SIZE, 
    test_size=TEST_SIZE, 
    stratify=y_bal, 
    random_state=42
)

# Scaling (-1 to 1 is optimal for Quantum Feature Maps)
scaler = MinMaxScaler(feature_range=(-1, 1))
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Data Prepared.")

# ==============================================================================
# 3. NOISE MODEL GENERATOR
# ==============================================================================
def create_noise_model():
    nm = NoiseModel()
    
    # 1. Single Qubit Errors (Thermal + Depolarizing)
    t1_err = thermal_relaxation_error(NOISE_CONFIG['T1'], NOISE_CONFIG['T2'], NOISE_CONFIG['GATE_TIME_1Q'])
    dep_err = depolarizing_error(NOISE_CONFIG['P_DEP_1Q'], 1)
    combined_1q = t1_err.compose(dep_err)
    nm.add_all_qubit_quantum_error(combined_1q, ['u1', 'u2', 'u3', 'rz', 'sx', 'x', 'id'])
    
    # 2. Two Qubit Errors (Thermal + Depolarizing)
    t2_q0 = thermal_relaxation_error(NOISE_CONFIG['T1'], NOISE_CONFIG['T2'], NOISE_CONFIG['GATE_TIME_2Q'])
    t2_q1 = thermal_relaxation_error(NOISE_CONFIG['T1'], NOISE_CONFIG['T2'], NOISE_CONFIG['GATE_TIME_2Q'])
    t2_err = t2_q0.tensor(t2_q1)
    dep_2q = depolarizing_error(NOISE_CONFIG['P_DEP_2Q'], 2)
    combined_2q = t2_err.compose(dep_2q)
    nm.add_all_qubit_quantum_error(combined_2q, ['cx', 'cz', 'ecr'])
    
    # 3. Readout Errors
    p_ro = NOISE_CONFIG['P_READOUT']
    ro_err = ReadoutError([[1 - p_ro, p_ro], [p_ro, 1 - p_ro]])
    nm.add_all_qubit_readout_error(ro_err)
    
    return nm

noise_model = create_noise_model()

# ==============================================================================
# 4. KERNEL COMPUTATION
# ==============================================================================
def get_feature_map(n_features, map_type):
    if map_type == 'Z':
        return ZFeatureMap(feature_dimension=n_features, reps=REPS)
    elif map_type == 'ZZ':
        return ZZFeatureMap(feature_dimension=n_features, reps=REPS, entanglement='full')
    elif map_type == 'Pauli_X_ZZ':
        return PauliFeatureMap(feature_dimension=n_features, reps=REPS, paulis=['X', 'ZZ'])
    else:
        raise ValueError("Invalid Map Type. Choose Z, ZZ, or Pauli_X_ZZ")

def compute_kernel_matrix(X1, X2, map_type):
    n_features = X1.shape[1]
    feature_map = get_feature_map(n_features, map_type)
    
    # Use Density Matrix simulator for accurate noise simulation
    simulator = AerSimulator(method='density_matrix', noise_model=noise_model)
    
    print(f" > Generating Quantum States for {len(X1)} + {len(X2)} samples...")
    
    def get_density_matrices(data):
        circuits = []
        for x in data:
            qc = feature_map.assign_parameters(x)
            qc.save_density_matrix()
            circuits.append(qc)
        
        # Run in batch
        transpiled = transpile(circuits, simulator)
        # shots=None forces exact calculation of the noisy density matrix
        result = simulator.run(transpiled, shots=None).result()
        
        return [DensityMatrix(result.data(i)['density_matrix']) for i in range(len(data))]

    # 1. Compute states for all data points
    states1 = get_density_matrices(X1)
    states2 = get_density_matrices(X2)
    
    # 2. Compute Fidelity (Gram Matrix)
    n1, n2 = len(X1), len(X2)
    K = np.zeros((n1, n2))
    
    print(f" > Calculating Fidelity Matrix ({n1}x{n2})...")
    start_k = time.time()
    
    for i in range(n1):
        for j in range(n2):
            K[i, j] = state_fidelity(states1[i], states2[j])
            
    print(f" > Matrix calculated in {time.time() - start_k:.2f}s")
    return K

# ==============================================================================
# 5. EXECUTION
# ==============================================================================
print(f"\nStarting Kernel Calculation...")
t_start = time.time()

# 1. Train Kernel
print("\n--- Computing Training Kernel ---")
K_train = compute_kernel_matrix(X_train, X_train, MAP_TYPE)

# 2. Test Kernel (Train vs Test)
print("\n--- Computing Testing Kernel ---")
K_test = compute_kernel_matrix(X_test, X_train, MAP_TYPE)

# 3. SVM Training
print("\n--- Training SVM ---")
svc = SVC(kernel='precomputed')
svc.fit(K_train, y_train)

# 4. Prediction
print("--- Predicting ---")
y_pred = svc.predict(K_test)

# Metrics Calculation
try:
    y_score = svc.decision_function(K_test)
    auc = roc_auc_score(y_test, y_score)
except:
    auc = 0.0
    print("Warning: Could not calculate AUC")

acc = accuracy_score(y_test, y_pred)

# --- NEW: Hit Rate and FAR Calculation ---
# cm structure: [[TN, FP], [FN, TP]]
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

# Hit Rate (Recall / Sensitivity): TP / (TP + FN)
hit_rate = tp / (tp + fn) if (tp + fn) > 0 else 0.0

# False Alarm Rate (False Positive Rate): FP / (FP + TN)
far = fp / (fp + tn) if (fp + tn) > 0 else 0.0

total_time = time.time() - t_start

# ==============================================================================
# 6. RESULTS
# ==============================================================================
print(f"\n{'='*60}")
print(f"FINAL RESULTS ({MAP_TYPE} Map | {len(FEATURE_LIST)} Features)")
print(f"{'='*60}")
print(f"Accuracy:       {acc:.4f}")
print(f"AUC Score:      {auc:.4f}")
print(f"Hit Rate (TPR): {hit_rate:.4f}")
print(f"False Alarm (FAR): {far:.4f}")
print(f"Total Time:     {total_time:.2f} seconds")
print(f"{'='*60}")


QSVM CONFIGURATION
Features:    ['V14', 'V7', 'V4', 'V19', 'V20', 'V17']
Map Type:    ZZ
Train Size:  400
Test Size:   200
Simulation:  Density Matrix with Thermal Noise
Data Prepared.

Starting Kernel Calculation...

--- Computing Training Kernel ---
 > Generating Quantum States for 400 + 400 samples...
 > Calculating Fidelity Matrix (400x400)...
 > Matrix calculated in 1518.45s

--- Computing Testing Kernel ---
 > Generating Quantum States for 200 + 400 samples...
 > Calculating Fidelity Matrix (200x400)...
 > Matrix calculated in 1140.69s

--- Training SVM ---
--- Predicting ---

FINAL RESULTS (ZZ Map | 6 Features)
Accuracy:       0.9050
AUC Score:      0.9755
Hit Rate (TPR): 0.8800
False Alarm (FAR): 0.0700
Total Time:     2669.99 seconds


In [2]:
#!/usr/bin/env python3
"""
QUANTUM KERNEL TRAINING WITH THERMAL NOISE
Config: 400 Train / 200 Test | Selectable Feature Maps | IBM Eagle Noise Profile
"""

import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix # <--- ADDED confusion_matrix
from sklearn.svm import SVC

# Qiskit Imports
from qiskit import transpile
from qiskit.circuit.library import ZZFeatureMap, ZFeatureMap, PauliFeatureMap
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, thermal_relaxation_error, depolarizing_error, ReadoutError
from qiskit.quantum_info import DensityMatrix, state_fidelity

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
DATA_PATH = 'creditcard.csv'

# --- Feature Selection ---
FEATURE_LIST = ['V14', 'V12', 'V4', 'V17', 'V8']

# --- Feature Map Selection ---
# Options: 'Z', 'ZZ', 'Pauli_X_ZZ'
MAP_TYPE = 'Pauli_X_ZZ'  
REPS = 1

# --- Dataset Size (Balanced) ---
TRAIN_SIZE = 400
TEST_SIZE = 200

# --- Hardware Noise Profile (IBM Eagle "Lower Quartile") ---
NOISE_CONFIG = {
    'T1': 60.0e3,         # 60 µs
    'T2': 30.0e3,         # 30 µs
    'GATE_TIME_1Q': 60.0, # 60 ns
    'GATE_TIME_2Q': 550.0,# 550 ns
    'P_DEP_1Q': 0.0004,   # 0.04%
    'P_DEP_2Q': 0.02,     # 2.0%
    'P_READOUT': 0.04     # 4.0%
}

print(f"\n{'='*60}")
print(f"QSVM CONFIGURATION")
print(f"{'='*60}")
print(f"Features:    {FEATURE_LIST}")
print(f"Map Type:    {MAP_TYPE}")
print(f"Train Size:  {TRAIN_SIZE}")
print(f"Test Size:   {TEST_SIZE}")
print(f"Simulation:  Density Matrix with Thermal Noise")

# ==============================================================================
# 2. DATA PREPARATION
# ==============================================================================
try:
    df = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    print(f"CRITICAL: {DATA_PATH} not found.")
    exit(1)

X_all = df[FEATURE_LIST].values
y_all = df['Class'].values

# Balance the Dataset (50/50 Fraud/Genuine)
fraud_idx = np.where(y_all == 1)[0]
genuine_idx = np.where(y_all == 0)[0]

# Ensure we have enough data
required_total = TRAIN_SIZE + TEST_SIZE
if len(fraud_idx) < required_total // 2:
    print(f"Warning: Not enough fraud samples for requested size. Using max available.")
    half_size = len(fraud_idx)
else:
    half_size = required_total // 2

indices = np.concatenate([
    np.random.choice(fraud_idx, half_size, replace=False),
    np.random.choice(genuine_idx, half_size, replace=False)
])
np.random.shuffle(indices)

X_bal = X_all[indices]
y_bal = y_all[indices]

# Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, 
    train_size=TRAIN_SIZE, 
    test_size=TEST_SIZE, 
    stratify=y_bal, 
    random_state=42
)

# Scaling (-1 to 1 is optimal for Quantum Feature Maps)
scaler = MinMaxScaler(feature_range=(-1, 1))
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Data Prepared.")

# ==============================================================================
# 3. NOISE MODEL GENERATOR
# ==============================================================================
def create_noise_model():
    nm = NoiseModel()
    
    # 1. Single Qubit Errors (Thermal + Depolarizing)
    t1_err = thermal_relaxation_error(NOISE_CONFIG['T1'], NOISE_CONFIG['T2'], NOISE_CONFIG['GATE_TIME_1Q'])
    dep_err = depolarizing_error(NOISE_CONFIG['P_DEP_1Q'], 1)
    combined_1q = t1_err.compose(dep_err)
    nm.add_all_qubit_quantum_error(combined_1q, ['u1', 'u2', 'u3', 'rz', 'sx', 'x', 'id'])
    
    # 2. Two Qubit Errors (Thermal + Depolarizing)
    t2_q0 = thermal_relaxation_error(NOISE_CONFIG['T1'], NOISE_CONFIG['T2'], NOISE_CONFIG['GATE_TIME_2Q'])
    t2_q1 = thermal_relaxation_error(NOISE_CONFIG['T1'], NOISE_CONFIG['T2'], NOISE_CONFIG['GATE_TIME_2Q'])
    t2_err = t2_q0.tensor(t2_q1)
    dep_2q = depolarizing_error(NOISE_CONFIG['P_DEP_2Q'], 2)
    combined_2q = t2_err.compose(dep_2q)
    nm.add_all_qubit_quantum_error(combined_2q, ['cx', 'cz', 'ecr'])
    
    # 3. Readout Errors
    p_ro = NOISE_CONFIG['P_READOUT']
    ro_err = ReadoutError([[1 - p_ro, p_ro], [p_ro, 1 - p_ro]])
    nm.add_all_qubit_readout_error(ro_err)
    
    return nm

noise_model = create_noise_model()

# ==============================================================================
# 4. KERNEL COMPUTATION
# ==============================================================================
def get_feature_map(n_features, map_type):
    if map_type == 'Z':
        return ZFeatureMap(feature_dimension=n_features, reps=REPS)
    elif map_type == 'ZZ':
        return ZZFeatureMap(feature_dimension=n_features, reps=REPS, entanglement='full')
    elif map_type == 'Pauli_X_ZZ':
        return PauliFeatureMap(feature_dimension=n_features, reps=REPS, paulis=['X', 'ZZ'])
    else:
        raise ValueError("Invalid Map Type. Choose Z, ZZ, or Pauli_X_ZZ")

def compute_kernel_matrix(X1, X2, map_type):
    n_features = X1.shape[1]
    feature_map = get_feature_map(n_features, map_type)
    
    # Use Density Matrix simulator for accurate noise simulation
    simulator = AerSimulator(method='density_matrix', noise_model=noise_model)
    
    print(f" > Generating Quantum States for {len(X1)} + {len(X2)} samples...")
    
    def get_density_matrices(data):
        circuits = []
        for x in data:
            qc = feature_map.assign_parameters(x)
            qc.save_density_matrix()
            circuits.append(qc)
        
        # Run in batch
        transpiled = transpile(circuits, simulator)
        # shots=None forces exact calculation of the noisy density matrix
        result = simulator.run(transpiled, shots=None).result()
        
        return [DensityMatrix(result.data(i)['density_matrix']) for i in range(len(data))]

    # 1. Compute states for all data points
    states1 = get_density_matrices(X1)
    states2 = get_density_matrices(X2)
    
    # 2. Compute Fidelity (Gram Matrix)
    n1, n2 = len(X1), len(X2)
    K = np.zeros((n1, n2))
    
    print(f" > Calculating Fidelity Matrix ({n1}x{n2})...")
    start_k = time.time()
    
    for i in range(n1):
        for j in range(n2):
            K[i, j] = state_fidelity(states1[i], states2[j])
            
    print(f" > Matrix calculated in {time.time() - start_k:.2f}s")
    return K

# ==============================================================================
# 5. EXECUTION
# ==============================================================================
print(f"\nStarting Kernel Calculation...")
t_start = time.time()

# 1. Train Kernel
print("\n--- Computing Training Kernel ---")
K_train = compute_kernel_matrix(X_train, X_train, MAP_TYPE)

# 2. Test Kernel (Train vs Test)
print("\n--- Computing Testing Kernel ---")
K_test = compute_kernel_matrix(X_test, X_train, MAP_TYPE)

# 3. SVM Training
print("\n--- Training SVM ---")
svc = SVC(kernel='precomputed')
svc.fit(K_train, y_train)

# 4. Prediction
print("--- Predicting ---")
y_pred = svc.predict(K_test)

# Metrics Calculation
try:
    y_score = svc.decision_function(K_test)
    auc = roc_auc_score(y_test, y_score)
except:
    auc = 0.0
    print("Warning: Could not calculate AUC")

acc = accuracy_score(y_test, y_pred)

# --- NEW: Hit Rate and FAR Calculation ---
# cm structure: [[TN, FP], [FN, TP]]
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

# Hit Rate (Recall / Sensitivity): TP / (TP + FN)
hit_rate = tp / (tp + fn) if (tp + fn) > 0 else 0.0

# False Alarm Rate (False Positive Rate): FP / (FP + TN)
far = fp / (fp + tn) if (fp + tn) > 0 else 0.0

total_time = time.time() - t_start

# ==============================================================================
# 6. RESULTS
# ==============================================================================
print(f"\n{'='*60}")
print(f"FINAL RESULTS ({MAP_TYPE} Map | {len(FEATURE_LIST)} Features)")
print(f"{'='*60}")
print(f"Accuracy:       {acc:.4f}")
print(f"AUC Score:      {auc:.4f}")
print(f"Hit Rate (TPR): {hit_rate:.4f}")
print(f"False Alarm (FAR): {far:.4f}")
print(f"Total Time:     {total_time:.2f} seconds")
print(f"{'='*60}")


QSVM CONFIGURATION
Features:    ['V14', 'V12', 'V4', 'V17', 'V8']
Map Type:    Pauli_X_ZZ
Train Size:  400
Test Size:   200
Simulation:  Density Matrix with Thermal Noise
Data Prepared.

Starting Kernel Calculation...

--- Computing Training Kernel ---
 > Generating Quantum States for 400 + 400 samples...
 > Calculating Fidelity Matrix (400x400)...
 > Matrix calculated in 586.40s

--- Computing Testing Kernel ---
 > Generating Quantum States for 200 + 400 samples...
 > Calculating Fidelity Matrix (200x400)...
 > Matrix calculated in 313.27s

--- Training SVM ---
--- Predicting ---

FINAL RESULTS (Pauli_X_ZZ Map | 5 Features)
Accuracy:       0.9100
AUC Score:      0.9600
Hit Rate (TPR): 0.9400
False Alarm (FAR): 0.1200
Total Time:     906.40 seconds


In [ ]:
#!/usr/bin/env python3
"""
QUANTUM KERNEL TRAINING WITH TRAPPED ION NOISE
Multi-Map Evaluation | Realistic Ion Trap Noise Profile
"""

import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.svm import SVC

# Qiskit Imports
from qiskit import transpile
from qiskit.circuit.library import ZZFeatureMap, ZFeatureMap, PauliFeatureMap
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, amplitude_damping_error, phase_damping_error, depolarizing_error, ReadoutError
from qiskit.quantum_info import DensityMatrix, state_fidelity

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
DATA_PATH = 'creditcard.csv'

# --- Feature Configurations for Each Map ---
FEATURE_CONFIGS = {
    'Z': ["V4", "V11", "V19", "V13", "V20"],
    'ZZ': ["V14", "V7", "V2", "V19", "V8", "V4"],
    'Pauli_X_ZZ': ["V17", "V21", "V2", "V1", "V14"]
}

REPS = 1

# --- Dataset Size (Balanced) ---
TRAIN_SIZE = 400
TEST_SIZE = 200

# --- Trapped Ion Noise Profile (Based on Real IonQ/Oxford Ionics Hardware) ---
# References:
# - IonQ Aria: 1Q: 99.94%, 2Q: 99.5% (production system)
# - IonQ Forte: 1Q: 99.96%, 2Q: 99.6% (30-qubit system, 2024)
# - Oxford Ionics: 1Q: 99.9992%, 2Q: 99.97% (lab record, 2024)
# - IonQ Harmony: 1Q: 99%, 2Q: 98% (older generation)
# 
# We use IonQ Aria specs as realistic "commercial" system:

NOISE_CONFIG = {
    # Single-qubit gate fidelity: 99.94% -> error rate 0.06%
    # This includes all errors: amplitude damping, dephasing, control errors
    'SINGLE_QUBIT_ERROR': 0.0006,
    
    # Two-qubit gate (Mølmer-Sørensen) fidelity: 99.5% -> error rate 0.5%
    # Main error sources: motional mode imperfections, crosstalk, laser intensity fluctuations
    'TWO_QUBIT_ERROR': 0.005,
    
    # Amplitude damping (spontaneous emission from metastable states)
    # T1 ~ 1-50 seconds for trapped ions (excellent!)
    # Contribution to gate error is minimal
    'AMPLITUDE_DAMPING_PROB': 0.00005,  # ~0.005% - very low
    
    # Phase damping (dephasing from magnetic field fluctuations, laser phase noise)
    # T2 ~ 0.2-2 seconds typically
    # Main contribution to single-qubit error
    'PHASE_DAMPING_PROB': 0.0003,  # ~0.03%
    
    # Readout error (state discrimination error)
    # Modern trapped ion systems: 99.5-99.9% fidelity
    # SPAM (State Preparation and Measurement) error
    'READOUT_ERROR': 0.003  # 0.3% - based on IonQ Aria specs
}

print(f"\n{'='*70}")
print(f"TRAPPED ION QSVM - MULTI-MAP EVALUATION")
print(f"{'='*70}")
print(f"Train Size:  {TRAIN_SIZE}")
print(f"Test Size:   {TEST_SIZE}")
print(f"Simulation:  Density Matrix with Trapped Ion Noise")
print(f"\nNoise Parameters (IonQ Aria-like Commercial System):")
print(f"  Single-Qubit Gate Error:  {NOISE_CONFIG['SINGLE_QUBIT_ERROR']*100:.3f}% (Fidelity: 99.94%)")
print(f"  Two-Qubit Gate Error:     {NOISE_CONFIG['TWO_QUBIT_ERROR']*100:.2f}% (Fidelity: 99.5%)")
print(f"  Amplitude Damping:        {NOISE_CONFIG['AMPLITUDE_DAMPING_PROB']*100:.4f}%")
print(f"  Phase Damping:            {NOISE_CONFIG['PHASE_DAMPING_PROB']*100:.3f}%")
print(f"  Readout Error:            {NOISE_CONFIG['READOUT_ERROR']*100:.2f}%")

# ==============================================================================
# 2. DATA PREPARATION
# ==============================================================================
try:
    df = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    print(f"\nCRITICAL: {DATA_PATH} not found.")
    exit(1)

# Get all unique features needed
all_features = set()
for features in FEATURE_CONFIGS.values():
    all_features.update(features)
all_features = list(all_features)

X_all = df[all_features + ['Class']].copy()
y_all = X_all['Class'].values
X_all = X_all.drop('Class', axis=1)

# Balance the Dataset (50/50 Fraud/Genuine)
fraud_idx = np.where(y_all == 1)[0]
genuine_idx = np.where(y_all == 0)[0]

required_total = TRAIN_SIZE + TEST_SIZE
half_size = required_total // 2

if len(fraud_idx) < half_size or len(genuine_idx) < half_size:
    print(f"\nERROR: Not enough samples in dataset.")
    print(f"Required: {half_size} fraud + {half_size} genuine = {required_total} total")
    print(f"Available: {len(fraud_idx)} fraud, {len(genuine_idx)} genuine")
    exit(1)

indices = np.concatenate([
    np.random.choice(fraud_idx, half_size, replace=False),
    np.random.choice(genuine_idx, half_size, replace=False)
])
np.random.shuffle(indices)

X_bal = X_all.iloc[indices]
y_bal = y_all[indices]

# Stratified Split
actual_train_size = min(TRAIN_SIZE, len(y_bal) - TEST_SIZE)
actual_test_size = min(TEST_SIZE, len(y_bal) - actual_train_size)

X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_bal, y_bal, 
    train_size=actual_train_size,
    test_size=actual_test_size,
    stratify=y_bal, 
    random_state=42
)

print(f"\nData Prepared: {len(y_train)} train, {len(y_test)} test samples")
print(f"Train class distribution: Fraud={sum(y_train==1)}, Genuine={sum(y_train==0)}")
print(f"Test class distribution:  Fraud={sum(y_test==1)}, Genuine={sum(y_test==0)}")

# ==============================================================================
# 3. TRAPPED ION NOISE MODEL
# ==============================================================================
def create_trapped_ion_noise_model():
    """
    Creates a noise model representative of trapped ion quantum computers.
    Key differences from superconducting qubits:
    - Much longer coherence times (T1 ~ seconds vs microseconds)
    - Primary errors: gate infidelity, phase noise, crosstalk
    - High-fidelity single-qubit gates
    - Lower fidelity two-qubit gates (Mølmer-Sørensen)
    """
    nm = NoiseModel()
    
    # 1. Single-Qubit Gate Errors
    # Combination of:
    # - Amplitude damping (spontaneous emission - very rare)
    # - Phase damping (magnetic field fluctuations)
    # - Small depolarizing component (other errors)
    
    amp_damp = amplitude_damping_error(NOISE_CONFIG['AMPLITUDE_DAMPING_PROB'])
    phase_damp = phase_damping_error(NOISE_CONFIG['PHASE_DAMPING_PROB'])
    depol_1q = depolarizing_error(NOISE_CONFIG['SINGLE_QUBIT_ERROR'] * 0.3, 1)
    
    # Compose errors
    single_qubit_error = amp_damp.compose(phase_damp).compose(depol_1q)
    
    # Apply to all single-qubit gates
    nm.add_all_qubit_quantum_error(single_qubit_error, 
                                   ['u1', 'u2', 'u3', 'rz', 'ry', 'rx', 'sx', 'x', 'id', 'h'])
    
    # 2. Two-Qubit Gate Errors (Mølmer-Sørensen gate)
    # These are typically less accurate than single-qubit gates
    # Main error sources: crosstalk, imperfect laser pulses
    
    # Apply amplitude and phase damping to both qubits
    amp_damp_2q = amp_damp.tensor(amp_damp)
    phase_damp_2q = phase_damp.tensor(phase_damp)
    depol_2q = depolarizing_error(NOISE_CONFIG['TWO_QUBIT_ERROR'], 2)
    
    two_qubit_error = amp_damp_2q.compose(phase_damp_2q).compose(depol_2q)
    
    # Apply to two-qubit gates
    nm.add_all_qubit_quantum_error(two_qubit_error, 
                                   ['cx', 'cz', 'cy', 'swap', 'iswap'])
    
    # 3. Readout Errors (state discrimination)
    # Trapped ions have excellent readout but not perfect
    p_ro = NOISE_CONFIG['READOUT_ERROR']
    # Asymmetric readout errors (|1⟩ -> |0⟩ more likely than |0⟩ -> |1⟩)
    ro_err = ReadoutError([[1 - p_ro, p_ro], 
                           [p_ro * 1.5, 1 - p_ro * 1.5]])
    nm.add_all_qubit_readout_error(ro_err)
    
    return nm

noise_model = create_trapped_ion_noise_model()

# ==============================================================================
# 4. KERNEL COMPUTATION
# ==============================================================================
def get_feature_map(n_features, map_type):
    if map_type == 'Z':
        return ZFeatureMap(feature_dimension=n_features, reps=REPS)
    elif map_type == 'ZZ':
        return ZZFeatureMap(feature_dimension=n_features, reps=REPS, entanglement='full')
    elif map_type == 'Pauli_X_ZZ':
        return PauliFeatureMap(feature_dimension=n_features, reps=REPS, paulis=['X', 'ZZ'])
    else:
        raise ValueError("Invalid Map Type")

def compute_kernel_matrix(X1, X2, feature_map):
    """Compute kernel matrix using trapped ion noise model"""
    simulator = AerSimulator(method='density_matrix', noise_model=noise_model)
    
    def get_density_matrices(data):
        circuits = []
        for x in data:
            qc = feature_map.assign_parameters(x)
            qc.save_density_matrix()
            circuits.append(qc)
        
        transpiled = transpile(circuits, simulator)
        result = simulator.run(transpiled, shots=None).result()
        
        return [DensityMatrix(result.data(i)['density_matrix']) for i in range(len(data))]

    states1 = get_density_matrices(X1)
    states2 = get_density_matrices(X2)
    
    n1, n2 = len(X1), len(X2)
    K = np.zeros((n1, n2))
    
    for i in range(n1):
        for j in range(n2):
            K[i, j] = state_fidelity(states1[i], states2[j])
            
    return K

# ==============================================================================
# 5. TRAIN AND EVALUATE EACH CONFIGURATION
# ==============================================================================
results = []

print(f"\n{'='*70}")
print(f"STARTING EVALUATIONS")
print(f"{'='*70}\n")

for map_type, feature_list in FEATURE_CONFIGS.items():
    print(f"\n{'─'*70}")
    print(f"MAP: {map_type} | FEATURES: {feature_list}")
    print(f"{'─'*70}")
    
    t_start = time.time()
    
    # 1. Prepare data for this feature set
    scaler = MinMaxScaler(feature_range=(-1, 1))
    X_train = scaler.fit_transform(X_train_full[feature_list])
    X_test = scaler.transform(X_test_full[feature_list])
    
    # 2. Create feature map
    n_features = len(feature_list)
    feature_map = get_feature_map(n_features, map_type)
    
    # 3. Compute kernels
    print(f"Computing Training Kernel ({len(X_train)}x{len(X_train)})...")
    K_train = compute_kernel_matrix(X_train, X_train, feature_map)
    
    print(f"Computing Testing Kernel ({len(X_test)}x{len(X_train)})...")
    K_test = compute_kernel_matrix(X_test, X_train, feature_map)
    
    # 4. Train SVM
    print(f"Training SVM...")
    svc = SVC(kernel='precomputed')
    svc.fit(K_train, y_train)
    
    # 5. Predict
    print(f"Predicting...")
    y_pred = svc.predict(K_test)
    
    # 6. Calculate Metrics
    try:
        y_score = svc.decision_function(K_test)
        auc = roc_auc_score(y_test, y_score)
    except:
        auc = 0.0
    
    acc = accuracy_score(y_test, y_pred)
    
    # Confusion Matrix: [[TN, FP], [FN, TP]]
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    hit_rate = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    far = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    
    total_time = time.time() - t_start
    
    # Store results
    results.append({
        'Map': map_type,
        'Features': len(feature_list),
        'Accuracy': acc,
        'AUC': auc,
        'Hit_Rate': hit_rate,
        'FAR': far,
        'Time': total_time
    })
    
    # Print individual results
    print(f"\n  ✓ Accuracy:         {acc:.4f}")
    print(f"  ✓ AUC Score:        {auc:.4f}")
    print(f"  ✓ Hit Rate (TPR):   {hit_rate:.4f}")
    print(f"  ✓ False Alarm Rate: {far:.4f}")
    print(f"  ✓ Time:             {total_time:.2f}s")

# ==============================================================================
# 6. SUMMARY TABLE
# ==============================================================================
print(f"\n{'='*70}")
print(f"FINAL SUMMARY - TRAPPED ION QSVM")
print(f"{'='*70}")
print(f"{'Map':<15} {'Features':<10} {'Accuracy':<10} {'AUC':<10} {'Hit Rate':<10} {'FAR':<10} {'Time (s)':<10}")
print(f"{'─'*70}")

for r in results:
    print(f"{r['Map']:<15} {r['Features']:<10} {r['Accuracy']:<10.4f} {r['AUC']:<10.4f} "
          f"{r['Hit_Rate']:<10.4f} {r['FAR']:<10.4f} {r['Time']:<10.1f}")

print(f"{'='*70}")

# Best performing configuration
best_acc = max(results, key=lambda x: x['Accuracy'])
best_auc = max(results, key=lambda x: x['AUC'])

print(f"\n🏆 Best Accuracy: {best_acc['Map']} ({best_acc['Accuracy']:.4f})")
print(f"🏆 Best AUC:      {best_auc['Map']} ({best_auc['AUC']:.4f})")
print(f"{'='*70}\n")


TRAPPED ION QSVM - MULTI-MAP EVALUATION
Train Size:  400
Test Size:   200
Simulation:  Density Matrix with Trapped Ion Noise

Noise Parameters (IonQ Aria-like Commercial System):
  Single-Qubit Gate Error:  0.060% (Fidelity: 99.94%)
  Two-Qubit Gate Error:     0.50% (Fidelity: 99.5%)
  Amplitude Damping:        0.0050%
  Phase Damping:            0.030%
  Readout Error:            0.30%

Data Prepared: 400 train, 200 test samples
Train class distribution: Fraud=200, Genuine=200
Test class distribution:  Fraud=100, Genuine=100

STARTING EVALUATIONS


──────────────────────────────────────────────────────────────────────
MAP: Z | FEATURES: ['V4', 'V11', 'V19', 'V13', 'V20']
──────────────────────────────────────────────────────────────────────
Computing Training Kernel (400x400)...
Computing Testing Kernel (200x400)...
Training SVM...
Predicting...

  ✓ Accuracy:         0.8750
  ✓ AUC Score:        0.9414
  ✓ Hit Rate (TPR):   0.8300
  ✓ False Alarm Rate: 0.0800
  ✓ Time:             8

In [ ]:
#!/usr/bin/env python3
"""
NOISE SENSITIVITY ANALYSIS - DEBUG TEST SCRIPT
Minimal test for multiprocessing validation
"""

import json
import logging
import multiprocessing
import os
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
from sklearn.svm import SVC

from qiskit import transpile
from qiskit.circuit.library import ZZFeatureMap
from qiskit_aer import AerSimulator
from qiskit_aer.noise import (
    NoiseModel, amplitude_damping_error, phase_damping_error,
    depolarizing_error, thermal_relaxation_error, ReadoutError
)
from qiskit.quantum_info import DensityMatrix, state_fidelity


# ==============================================================================
# SIMPLIFIED BASELINE PARAMETERS
# ==============================================================================
BASELINE_SUPERCONDUCTING = {
    'T1': 50e-6,
    'T2': 70e-6,
    'GATE_TIME_1Q': 35e-9,
    'GATE_TIME_2Q': 300e-9,
    'P_DEP_1Q': 0.001,
    'P_DEP_2Q': 0.01,
    'P_READOUT': 0.02
}

BASELINE_TRAPPED_ION = {
    'SINGLE_QUBIT_ERROR': 0.0006,
    'TWO_QUBIT_ERROR': 0.005,
    'AMPLITUDE_DAMPING_PROB': 0.00005,
    'PHASE_DAMPING_PROB': 0.0003,
    'READOUT_ERROR': 0.003
}

# ==============================================================================
# TEST CONFIGURATION (MINIMAL)
# ==============================================================================
DATA_PATH = 'creditcard.csv'
TRAIN_SIZE = 50  # Reduced for speed
TEST_SIZE = 30   # Reduced for speed

# Test only ONE parameter with 5 values
TEST_SWEEP_SUPERCONDUCTING = {
    'P_DEP_1Q': [0.0005, 0.00075, 0.001, 0.00125, 0.0015]
}

TEST_SWEEP_TRAPPED_ION = {
    'TWO_QUBIT_ERROR': [0.003, 0.004, 0.005, 0.006, 0.007]
}

# Simple feature subsets
TEST_FEATURES = {
    'trapped_ion': ['V14', 'V7', 'V2', 'V19', 'V8'],
    'superconducting': ['V14', 'V7', 'V4', 'V19', 'V20']
}


# ==============================================================================
# LOGGING
# ==============================================================================
def setup_logging():
    log_dir = 'debug_logs'
    os.makedirs(log_dir, exist_ok=True)
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    log_file = os.path.join(log_dir, f'debug_test_{timestamp}.log')
    
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s | %(levelname)-8s | %(processName)-15s | %(message)s',
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )
    
    return log_file


# ==============================================================================
# DATA LOADING
# ==============================================================================
def load_test_data():
    logger = logging.getLogger(__name__)
    logger.info("Loading test dataset...")
    
    try:
        df = pd.read_csv(DATA_PATH)
    except FileNotFoundError:
        logger.error(f"CRITICAL: {DATA_PATH} not found")
        raise
    
    # Get features
    features = list(set(TEST_FEATURES['trapped_ion'] + TEST_FEATURES['superconducting']))
    X_all = df[features + ['Class']].copy()
    y_all = X_all['Class'].values
    X_all = X_all.drop('Class', axis=1)
    
    # Balance dataset
    fraud_idx = np.where(y_all == 1)[0]
    genuine_idx = np.where(y_all == 0)[0]
    
    required_total = TRAIN_SIZE + TEST_SIZE
    half_size = required_total // 2
    
    np.random.seed(42)
    indices = np.concatenate([
        np.random.choice(fraud_idx, half_size, replace=False),
        np.random.choice(genuine_idx, half_size, replace=False)
    ])
    np.random.shuffle(indices)
    
    X_bal = X_all.iloc[indices]
    y_bal = y_all[indices]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_bal, y_bal,
        train_size=TRAIN_SIZE,
        test_size=TEST_SIZE,
        stratify=y_bal,
        random_state=42
    )
    
    logger.info(f"Loaded: {len(y_train)} train, {len(y_test)} test samples")
    return X_train, X_test, y_train, y_test


# ==============================================================================
# NOISE MODELS
# ==============================================================================
def create_superconducting_noise_model(params):
    nm = NoiseModel()
    
    # Single-qubit
    t_relax_1q = thermal_relaxation_error(
        params['T1'], params['T2'], params['GATE_TIME_1Q']
    )
    nm.add_all_qubit_quantum_error(t_relax_1q, ['rz', 'ry', 'rx', 'sx', 'x', 'h'])
    
    dep_1q = depolarizing_error(params['P_DEP_1Q'], 1)
    nm.add_all_qubit_quantum_error(dep_1q, ['rz', 'ry', 'rx', 'sx', 'x', 'h'])
    
    # Two-qubit
    t_relax_2q = thermal_relaxation_error(
        params['T1'], params['T2'], params['GATE_TIME_2Q']
    ).tensor(thermal_relaxation_error(
        params['T1'], params['T2'], params['GATE_TIME_2Q']
    ))
    nm.add_all_qubit_quantum_error(t_relax_2q, ['cx'])
    
    dep_2q = depolarizing_error(params['P_DEP_2Q'], 2)
    nm.add_all_qubit_quantum_error(dep_2q, ['cx'])
    
    # Readout
    ro_err = ReadoutError([
        [1 - params['P_READOUT'], params['P_READOUT']],
        [params['P_READOUT'], 1 - params['P_READOUT']]
    ])
    nm.add_all_qubit_readout_error(ro_err)
    
    return nm


def create_trapped_ion_noise_model(params):
    nm = NoiseModel()
    
    # Single-qubit
    amp_damp = amplitude_damping_error(params['AMPLITUDE_DAMPING_PROB'])
    phase_damp = phase_damping_error(params['PHASE_DAMPING_PROB'])
    depol_1q = depolarizing_error(params['SINGLE_QUBIT_ERROR'] * 0.3, 1)
    
    single_qubit_error = amp_damp.compose(phase_damp).compose(depol_1q)
    nm.add_all_qubit_quantum_error(single_qubit_error, ['rz', 'ry', 'rx', 'sx', 'x', 'h'])
    
    # Two-qubit
    amp_damp_2q = amp_damp.tensor(amp_damp)
    phase_damp_2q = phase_damp.tensor(phase_damp)
    depol_2q = depolarizing_error(params['TWO_QUBIT_ERROR'], 2)
    
    two_qubit_error = amp_damp_2q.compose(phase_damp_2q).compose(depol_2q)
    nm.add_all_qubit_quantum_error(two_qubit_error, ['cx'])
    
    # Readout
    p_ro = params['READOUT_ERROR']
    ro_err = ReadoutError([
        [1 - p_ro, p_ro],
        [p_ro * 1.5, 1 - p_ro * 1.5]
    ])
    nm.add_all_qubit_readout_error(ro_err)
    
    return nm


# ==============================================================================
# KERNEL COMPUTATION
# ==============================================================================
def compute_kernel_matrix(X1, X2, feature_map, noise_model):
    simulator = AerSimulator(method='density_matrix', noise_model=noise_model)
    
    def get_density_matrices(data):
        circuits = []
        for x in data:
            qc = feature_map.assign_parameters(x)
            qc.save_density_matrix()
            circuits.append(qc)
        
        transpiled = transpile(circuits, simulator)
        result = simulator.run(transpiled, shots=None).result()
        
        return [DensityMatrix(result.data(i)['density_matrix']) for i in range(len(data))]
    
    states1 = get_density_matrices(X1)
    states2 = get_density_matrices(X2)
    
    n1, n2 = len(X1), len(X2)
    K = np.zeros((n1, n2))
    
    for i in range(n1):
        for j in range(n2):
            K[i, j] = state_fidelity(states1[i], states2[j])
    
    return K


# ==============================================================================
# TEST WORKER
# ==============================================================================
def test_worker(worker_id, architecture, feature_list, baseline_params,
                sweep_param, sweep_values, X_train, X_test, y_train, y_test,
                output_file):
    logger = logging.getLogger(__name__)
    
    logger.info("="*70)
    logger.info(f"TEST WORKER {worker_id} STARTED")
    logger.info(f"Architecture: {architecture}")
    logger.info(f"Features: {feature_list}")
    logger.info(f"Sweep Parameter: {sweep_param}")
    logger.info(f"Sweep Values: {sweep_values}")
    logger.info("="*70)
    
    results = []
    
    # Scale data
    scaler = MinMaxScaler(feature_range=(-1, 1))
    X_train_scaled = scaler.fit_transform(X_train[feature_list])
    X_test_scaled = scaler.transform(X_test[feature_list])
    
    # Create feature map (always ZZFeatureMap for simplicity)
    n_features = len(feature_list)
    feature_map = ZZFeatureMap(feature_dimension=n_features, reps=1, entanglement='full')
    
    logger.info(f"Worker {worker_id}: Created ZZFeatureMap with {n_features} features")
    
    # Sweep parameter
    for value in sweep_values:
        noise_params = baseline_params.copy()
        noise_params[sweep_param] = value
        
        logger.info(f"Worker {worker_id}: Testing {sweep_param} = {value}")
        
        try:
            # Create noise model
            if architecture == 'superconducting':
                noise_model = create_superconducting_noise_model(noise_params)
            else:
                noise_model = create_trapped_ion_noise_model(noise_params)
            
            logger.info(f"Worker {worker_id}: Noise model created")
            
            # Compute kernels
            logger.info(f"Worker {worker_id}: Computing training kernel...")
            K_train = compute_kernel_matrix(X_train_scaled, X_train_scaled,
                                           feature_map, noise_model)
            
            logger.info(f"Worker {worker_id}: Computing testing kernel...")
            K_test = compute_kernel_matrix(X_test_scaled, X_train_scaled,
                                          feature_map, noise_model)
            
            # Train SVM
            logger.info(f"Worker {worker_id}: Training SVM...")
            svc = SVC(kernel='precomputed')
            svc.fit(K_train, y_train)
            
            # Predict and compute AUC
            y_pred = svc.predict(K_test)
            try:
                y_score = svc.decision_function(K_test)
                auc = roc_auc_score(y_test, y_score)
            except:
                auc = 0.0
            
            logger.info(f"Worker {worker_id}: {sweep_param}={value} -> AUC={auc:.4f}")
            
            # Store result
            result_entry = {
                'worker_id': worker_id,
                'architecture': architecture,
                'feature_list': feature_list,
                'baseline_params': baseline_params,
                'swept_parameter': sweep_param,
                'parameter_value': float(value),
                'auc': auc
            }
            results.append(result_entry)
            
        except Exception as e:
            logger.error(f"Worker {worker_id}: ERROR at {sweep_param}={value}: {str(e)}")
    
    # Save JSON
    try:
        with open(output_file, 'w') as f:
            json.dump(results, f, indent=2)
        logger.info(f"Worker {worker_id}: Saved results to {output_file}")
    except Exception as e:
        logger.error(f"Worker {worker_id}: Failed to save JSON: {str(e)}")
    
    logger.info(f"Worker {worker_id}: COMPLETED")
    return output_file


# ==============================================================================
# MAIN
# ==============================================================================
def main():
    log_file = setup_logging()
    logger = logging.getLogger(__name__)
    
    logger.info("="*70)
    logger.info("DEBUG TEST SCRIPT - NOISE SENSITIVITY ANALYSIS")
    logger.info("="*70)
    logger.info(f"Log file: {log_file}")
    
    # Log baselines
    logger.info("")
    logger.info("BASELINE PARAMETERS:")
    logger.info(f"Superconducting: {BASELINE_SUPERCONDUCTING}")
    logger.info(f"Trapped-Ion: {BASELINE_TRAPPED_ION}")
    logger.info("")
    
    # Load data
    X_train, X_test, y_train, y_test = load_test_data()
    
    # Create output directory
    output_dir = 'debug_results'
    os.makedirs(output_dir, exist_ok=True)
    
    # Configure 6 test workers (3 per architecture)
    worker_configs = []
    
    # Trapped-Ion: 3 workers testing the same parameter
    for i in range(3):
        worker_id = i + 1
        output_file = os.path.join(output_dir, f'debug_trapped_ion_worker{worker_id}.json')
        worker_configs.append((
            worker_id, 'trapped_ion', TEST_FEATURES['trapped_ion'],
            BASELINE_TRAPPED_ION, 'TWO_QUBIT_ERROR',
            TEST_SWEEP_TRAPPED_ION['TWO_QUBIT_ERROR'],
            X_train, X_test, y_train, y_test, output_file
        ))
    
    # Superconducting: 3 workers testing the same parameter
    for i in range(3):
        worker_id = i + 4
        output_file = os.path.join(output_dir, f'debug_superconducting_worker{worker_id}.json')
        worker_configs.append((
            worker_id, 'superconducting', TEST_FEATURES['superconducting'],
            BASELINE_SUPERCONDUCTING, 'P_DEP_1Q',
            TEST_SWEEP_SUPERCONDUCTING['P_DEP_1Q'],
            X_train, X_test, y_train, y_test, output_file
        ))
    
    logger.info(f"Configured {len(worker_configs)} test workers")
    
    # Launch multiprocessing
    logger.info("="*70)
    logger.info("LAUNCHING 6 TEST WORKERS")
    logger.info("="*70)
    
    with multiprocessing.Pool(processes=6) as pool:
        results = pool.starmap(test_worker, worker_configs)
    
    logger.info("="*70)
    logger.info("ALL TEST WORKERS COMPLETED")
    logger.info("="*70)
    logger.info("Output files:")
    for r in results:
        logger.info(f"  - {r}")
    
    logger.info("="*70)
    logger.info("DEBUG TEST COMPLETE - CHECK LOGS FOR ANY ERRORS")
    logger.info("="*70)


if __name__ == "__main__":
    main()